In [1]:
from sklearn.svm import SVR
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score



In [2]:
df=datasets.load_diabetes(as_frame=True).frame
df

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0
...,...,...,...,...,...,...,...,...,...,...,...
437,0.041708,0.050680,0.019662,0.059744,-0.005697,-0.002566,-0.028674,-0.002592,0.031193,0.007207,178.0
438,-0.005515,0.050680,-0.015906,-0.067642,0.049341,0.079165,-0.028674,0.034309,-0.018114,0.044485,104.0
439,0.041708,0.050680,-0.015906,0.017293,-0.037344,-0.013840,-0.024993,-0.011080,-0.046883,0.015491,132.0
440,-0.045472,-0.044642,0.039062,0.001215,0.016318,0.015283,-0.028674,0.026560,0.044529,-0.025930,220.0


In [3]:
X=df.drop("target",axis=1)
Y=df["target"]

In [4]:
X_train,X_test,Y_train,Y_test=train_test_split(
    X,
    Y,
    test_size=0.3,
    random_state=42,
    
    
)

In [5]:
# scale our target values
scaler=StandardScaler()
Y_train_scaled=scaler.fit_transform(Y_train.values.reshape(-1,1)).ravel()
Y_test_scaled=scaler.transform(Y_test.values.reshape(-1,1)).ravel()



In [6]:
svr=SVR()
svr.fit(X_train,Y_train_scaled)


,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,tol,0.001
,C,1.0
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [7]:
y_pre_scaled=svr.predict(X_test)

In [8]:
print("r2_score",r2_score(Y_test_scaled,y_pre_scaled)*100,"%")

r2_score 48.84444315165189 %


In [9]:
y_pre_scaled=svr.predict(X_test)
Y_train_pred_scaled=svr.predict(X_train)

print("r2_score",r2_score(Y_test_scaled,y_pre_scaled)*100,"%")
print("r2_score",r2_score(Y_train_scaled,Y_train_pred_scaled)*100,"%")

r2_score 48.84444315165189 %
r2_score 65.96361676267712 %


# linear kernel

In [10]:
# model
svr=SVR(kernel="linear")
svr.fit(X_train,Y_train_scaled)

y_pre_scaled=svr.predict(X_test)
Y_train_pred_scaled=svr.predict(X_train)

print("r2_score",r2_score(Y_test_scaled,y_pre_scaled)*100,"%")
print("r2_score",r2_score(Y_train_scaled,Y_train_pred_scaled)*100,"%")



r2_score 44.33761323833776 %
r2_score 45.19122998247524 %


# polynomial Kernel

In [11]:
# model
svr=SVR(kernel="poly")
svr.fit(X_train,Y_train_scaled)

y_pre_scaled=svr.predict(X_test)
Y_train_pred_scaled=svr.predict(X_train)

print("r2_score",r2_score(Y_test_scaled,y_pre_scaled)*100,"%")
print("r2_score",r2_score(Y_train_scaled,Y_train_pred_scaled)*100,"%")



r2_score 24.203771038107757 %
r2_score 57.90920834310542 %


# sigmoid kernel

In [12]:
# model
svr=SVR(kernel="sigmoid")
svr.fit(X_train,Y_train_scaled)

y_pre_scaled=svr.predict(X_test)
Y_train_pred_scaled=svr.predict(X_train)

print("r2_score",r2_score(Y_test_scaled,y_pre_scaled)*100,"%")
print("r2_score",r2_score(Y_train_scaled,Y_train_pred_scaled)*100,"%")



r2_score -1531.6808189576823 %
r2_score -1972.1193440731313 %


# hyperparameter tuning using Gridsearchcv

In [14]:
from sklearn.model_selection import GridSearchCV

param_grid={
    "C" : [1,2,5,10,50,100],
    "kernel" : ["rbf","linear"],
    "epsilon" : [0.01,0.1,0.2,0.3,]
}

svr=SVR()

grid_search=GridSearchCV(svr,param_grid,scoring="r2",cv=5)
grid_search.fit(X_train,Y_train_scaled)


print("best_params",grid_search.best_params_)



best_params {'C': 10, 'epsilon': 0.1, 'kernel': 'linear'}


In [15]:
best_model=SVR(kernel="linear",C=10,epsilon=0.1)
best_model.fit(X_train,Y_train_scaled)

y_pre_scaled=best_model.predict(X_test)
Y_train_pred_scaled=best_model.predict(X_train)

print("r2_score",r2_score(Y_test_scaled,y_pre_scaled)*100,"%")
print("r2_score",r2_score(Y_train_scaled,Y_train_pred_scaled)*100,"%")

r2_score 47.444183250401096 %
r2_score 51.51066486918876 %


# linear svr with no kernel parameter

In [19]:
from sklearn.svm import LinearSVR


svr=LinearSVR(C=10,epsilon=0.1,max_iter=5000)
svr.fit(X_train,Y_train_scaled)

y_pre_scaled=svr.predict(X_test)
Y_train_pred_scaled=svr.predict(X_train)

print("r2_score",r2_score(Y_test_scaled,y_pre_scaled)*100,"%")
print("r2_score",r2_score(Y_train_scaled,Y_train_pred_scaled)*100,"%")

r2_score 47.448846543926 %
r2_score 51.51593845904729 %
